In [38]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

In [39]:
import os
import pandas as pd
import pgserver

from sqlalchemy import create_engine, text

pg = pgserver.get_server(
    '/tmp/hw5_pg',
    cleanup_mode='stop'
)

engine = create_engine(
    pg.get_uri(),
    future=True
)

with engine.connect() as conn:
    version = conn.execute(
        text("SELECT version()")
    ).scalar_one()

print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [40]:
import kagglehub

dataset_dir = kagglehub.dataset_download(
    'olistbr/brazilian-ecommerce'
)

print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [41]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)

    df = pd.read_csv(path)

    df.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=10_000
    )

    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [42]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS olist_order_items CASCADE;
        DROP TABLE IF EXISTS olist_orders CASCADE;
        DROP TABLE IF EXISTS olist_customers CASCADE;
        DROP TABLE IF EXISTS olist_products CASCADE;
        DROP TABLE IF EXISTS olist_sellers CASCADE;

        CREATE TABLE olist_customers (
            customer_id TEXT PRIMARY KEY,
            customer_unique_id TEXT NOT NULL,
            customer_zip_code_prefix INTEGER,
            customer_city TEXT,
            customer_state CHAR(2) NOT NULL
        );

        CREATE TABLE olist_orders (
            order_id TEXT PRIMARY KEY,
            customer_id TEXT NOT NULL
                REFERENCES olist_customers(customer_id),

            order_status TEXT NOT NULL,

            order_purchase_timestamp TIMESTAMP NOT NULL,
            order_approved_at TIMESTAMP,
            order_delivered_carrier_date TIMESTAMP,
            order_delivered_customer_date TIMESTAMP,
            order_estimated_delivery_date TIMESTAMP,

            CHECK (
                order_status IN (
                    'created',
                    'approved',
                    'invoiced',
                    'processing',
                    'shipped',
                    'delivered',
                    'unavailable',
                    'canceled'
                )
            )
        );
    """))

print("customers and orders tables created!")


customers and orders tables created!


In [43]:
pd.read_sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""", engine)

,table_name
0,hw5_referrals
1,olist_customers
2,olist_customers_raw
3,olist_order_items_raw
4,olist_orders
5,olist_orders_raw
6,olist_products_raw
7,olist_sellers_raw


In [44]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE olist_products (
            product_id TEXT PRIMARY KEY,
            product_category_name TEXT,

            product_name_length INTEGER
                CHECK (
                    product_name_length IS NULL
                    OR product_name_length >= 0
                ),

            product_description_length INTEGER
                CHECK (
                    product_description_length IS NULL
                    OR product_description_length >= 0
                ),

            product_photos_qty INTEGER
                CHECK (
                    product_photos_qty IS NULL
                    OR product_photos_qty >= 0
                ),

            product_weight_g INTEGER
                CHECK (
                    product_weight_g IS NULL
                    OR product_weight_g >= 0
                ),

            product_length_cm INTEGER
                CHECK (
                    product_length_cm IS NULL
                    OR product_length_cm >= 0
                ),

            product_height_cm INTEGER
                CHECK (
                    product_height_cm IS NULL
                    OR product_height_cm >= 0
                ),

            product_width_cm INTEGER
                CHECK (
                    product_width_cm IS NULL
                    OR product_width_cm >= 0
                )
        );

        CREATE TABLE olist_sellers (
            seller_id TEXT PRIMARY KEY,
            seller_zip_code_prefix INTEGER,
            seller_city TEXT,
            seller_state CHAR(2) NOT NULL
        );
    """))

print("products and sellers created!")

products and sellers created!


In [45]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO olist_products
        SELECT
            product_id,
            product_category_name,
            product_name_lenght::INTEGER,
            product_description_lenght::INTEGER,
            product_photos_qty::INTEGER,
            product_weight_g::INTEGER,
            product_length_cm::INTEGER,
            product_height_cm::INTEGER,
            product_width_cm::INTEGER
        FROM olist_products_raw;

        INSERT INTO olist_sellers
        SELECT
            seller_id,
            seller_zip_code_prefix::INTEGER,
            seller_city,
            seller_state::CHAR(2)
        FROM olist_sellers_raw;
    """))

print("products and sellers loaded!")

products and sellers loaded!


In [46]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS olist_order_items CASCADE;

        CREATE TABLE olist_order_items (
            order_id TEXT NOT NULL
                REFERENCES olist_orders(order_id),

            order_item_id INTEGER NOT NULL,

            product_id TEXT
                REFERENCES olist_products(product_id),

            seller_id TEXT
                REFERENCES olist_sellers(seller_id),

            shipping_limit_date TIMESTAMP,

            price NUMERIC(12, 2)
                CHECK (price IS NULL OR price >= 0),

            freight_value NUMERIC(12, 2)
                CHECK (freight_value IS NULL OR freight_value >= 0),

            PRIMARY KEY (order_id, order_item_id)
        );
    """))

print("order_items created!")

order_items created!


In [47]:
check_counts = pd.read_sql("""
    SELECT 'customers_raw' AS table_name, COUNT(*) AS n_rows
    FROM olist_customers_raw

    UNION ALL
    SELECT 'customers_typed', COUNT(*)
    FROM olist_customers

    UNION ALL
    SELECT 'orders_raw', COUNT(*)
    FROM olist_orders_raw

    UNION ALL
    SELECT 'orders_typed', COUNT(*)
    FROM olist_orders;
""", engine)

check_counts

,table_name,n_rows
0,orders_typed,0
1,customers_typed,0
2,customers_raw,99441
3,orders_raw,99441


In [48]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO olist_customers
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix::INTEGER,
            customer_city,
            customer_state::CHAR(2)
        FROM olist_customers_raw
        ON CONFLICT (customer_id) DO NOTHING;
    """))

print("customers loaded!")

customers loaded!


In [49]:
pd.read_sql("""
    SELECT COUNT(*) AS customers
    FROM olist_customers;
""", engine)

,customers
0,99441


In [50]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO olist_orders
        SELECT
            order_id,
            customer_id,
            order_status,
            order_purchase_timestamp::TIMESTAMP,
            order_approved_at::TIMESTAMP,
            order_delivered_carrier_date::TIMESTAMP,
            order_delivered_customer_date::TIMESTAMP,
            order_estimated_delivery_date::TIMESTAMP
        FROM olist_orders_raw
        ON CONFLICT (order_id) DO NOTHING;
    """))

print("orders loaded!")

orders loaded!


In [51]:
pd.read_sql("""
    SELECT COUNT(*) AS orders
    FROM olist_orders;
""", engine)

,orders
0,99441


In [52]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO olist_order_items
        SELECT
            order_id,
            order_item_id::INTEGER,
            product_id,
            seller_id,
            shipping_limit_date::TIMESTAMP,
            price::NUMERIC(12, 2),
            freight_value::NUMERIC(12, 2)
        FROM olist_order_items_raw
        ON CONFLICT (order_id, order_item_id) DO NOTHING;
    """))

print("order_items loaded!")

order_items loaded!


In [53]:
smoke_test = pd.read_sql("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows
    FROM olist_customers

    UNION ALL
    SELECT 'orders', COUNT(*)
    FROM olist_orders

    UNION ALL
    SELECT 'order_items', COUNT(*)
    FROM olist_order_items

    UNION ALL
    SELECT 'products', COUNT(*)
    FROM olist_products

    UNION ALL
    SELECT 'sellers', COUNT(*)
    FROM olist_sellers

    ORDER BY table_name;
""", engine)

smoke_test

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095


In [54]:
smoke_test = pd.read_sql("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows
    FROM olist_customers

    UNION ALL
    SELECT 'orders', COUNT(*)
    FROM olist_orders

    UNION ALL
    SELECT 'order_items', COUNT(*)
    FROM olist_order_items

    UNION ALL
    SELECT 'products', COUNT(*)
    FROM olist_products

    UNION ALL
    SELECT 'sellers', COUNT(*)
    FROM olist_sellers

    ORDER BY table_name;
""", engine)

smoke_test

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095


In [55]:
stage2 = pd.read_sql("""
    WITH

    reference_date AS (
        -- Stage 0: stable reference date
        SELECT
            (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
        FROM olist_orders
    ),

    stage1_orders_total AS (
        -- Stage 1: orders + order_total
        SELECT
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp AS order_ts,
            o.order_purchase_timestamp::DATE AS order_date,

            COALESCE(
                SUM(
                    COALESCE(oi.price, 0)
                    + COALESCE(oi.freight_value, 0)
                ),
                0
            )::NUMERIC(12, 2) AS order_total

        FROM olist_orders AS o

        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id

        GROUP BY
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp
    ),

    stage2_rfm AS (
        -- Stage 2: RFM features per customer
        SELECT
            s.customer_id,

            (
                rd.as_of_date - MAX(s.order_date)
            )::INTEGER AS recency_days,

            COUNT(*) AS frequency,

            SUM(s.order_total)::NUMERIC(12, 2) AS monetary,

            MIN(s.order_date) AS first_order_date

        FROM stage1_orders_total AS s

        CROSS JOIN reference_date AS rd

        GROUP BY
            s.customer_id,
            rd.as_of_date
    )

    SELECT *
    FROM stage2_rfm
    ORDER BY monetary DESC
    LIMIT 10;
""", engine)

stage2

,customer_id,recency_days,frequency,monetary,first_order_date
0,1617b1357756262bfa56ab541c47bc16,384,1,13664.08,2017-09-29
1,ec5b2ba62e574342386871631fafd3fc,95,1,7274.88,2018-07-15
2,c6e2731c5b391845f6800c97401a43a9,613,1,6929.31,2017-02-12
3,f48d464a0baaea338cb25f816991ab1f,85,1,6922.21,2018-07-25
4,3fd6777bbce08a352fddd04e4a7cc8f6,512,1,6726.66,2017-05-24
5,05455dfa7cd02f13d132aa7a6a9729c6,328,1,6081.54,2017-11-24
6,df55c14d1476a9a3467f131269c2477f,565,1,4950.34,2017-04-01
7,e0a2412720e9ea4f26c1ac985f6a7358,98,1,4809.44,2018-07-12
8,24bbf5fd2f2e1b359ee7de94defc4a15,548,1,4764.34,2017-04-18
9,3d979689f636322c62418b6346b1c6d2,118,1,4681.78,2018-06-22


In [56]:
stage3 = pd.read_sql("""
    WITH

    reference_date AS (
        -- Stage 0: stable reference date
        SELECT
            (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
        FROM olist_orders
    ),

    stage1_orders_total AS (
        -- Stage 1: orders + order_total
        SELECT
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp AS order_ts,
            o.order_purchase_timestamp::DATE AS order_date,

            COALESCE(
                SUM(
                    COALESCE(oi.price, 0)
                    + COALESCE(oi.freight_value, 0)
                ),
                0
            )::NUMERIC(12, 2) AS order_total

        FROM olist_orders AS o

        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id

        GROUP BY
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp
    ),

    stage2_rfm AS (
        -- Stage 2: RFM features
        SELECT
            s.customer_id,
            (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
            COUNT(*) AS frequency,
            SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
            MIN(s.order_date) AS first_order_date

        FROM stage1_orders_total AS s

        CROSS JOIN reference_date AS rd

        GROUP BY
            s.customer_id,
            rd.as_of_date
    ),

    stage3_cohort AS (
        -- Stage 3: cohort quarter
        SELECT
            r.*,

            DATE_TRUNC(
                'quarter',
                r.first_order_date
            )::DATE AS cohort_quarter

        FROM stage2_rfm AS r
    )

    SELECT *
    FROM stage3_cohort
    ORDER BY monetary DESC
    LIMIT 10;
""", engine)

stage3

,customer_id,recency_days,frequency,monetary,first_order_date,cohort_quarter
0,1617b1357756262bfa56ab541c47bc16,384,1,13664.08,2017-09-29,2017-07-01
1,ec5b2ba62e574342386871631fafd3fc,95,1,7274.88,2018-07-15,2018-07-01
2,c6e2731c5b391845f6800c97401a43a9,613,1,6929.31,2017-02-12,2017-01-01
3,f48d464a0baaea338cb25f816991ab1f,85,1,6922.21,2018-07-25,2018-07-01
4,3fd6777bbce08a352fddd04e4a7cc8f6,512,1,6726.66,2017-05-24,2017-04-01
5,05455dfa7cd02f13d132aa7a6a9729c6,328,1,6081.54,2017-11-24,2017-10-01
6,df55c14d1476a9a3467f131269c2477f,565,1,4950.34,2017-04-01,2017-04-01
7,e0a2412720e9ea4f26c1ac985f6a7358,98,1,4809.44,2018-07-12,2018-07-01
8,24bbf5fd2f2e1b359ee7de94defc4a15,548,1,4764.34,2017-04-18,2017-04-01
9,3d979689f636322c62418b6346b1c6d2,118,1,4681.78,2018-06-22,2018-04-01


In [57]:
stage4 = pd.read_sql("""
    WITH

    reference_date AS (
        -- Stage 0: stable reference date
        SELECT
            (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
        FROM olist_orders
    ),

    stage1_orders_total AS (
        -- Stage 1: orders + order_total
        SELECT
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp AS order_ts,
            o.order_purchase_timestamp::DATE AS order_date,

            COALESCE(
                SUM(
                    COALESCE(oi.price, 0)
                    + COALESCE(oi.freight_value, 0)
                ),
                0
            )::NUMERIC(12, 2) AS order_total

        FROM olist_orders AS o

        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id

        GROUP BY
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp
    ),

    stage4_rolling_aov AS (
        -- Stage 4: rolling 30-day average order value
        SELECT
            s.customer_id,
            s.order_id,
            s.order_ts,
            s.order_total,

            AVG(s.order_total) OVER (
                PARTITION BY s.customer_id
                ORDER BY s.order_ts
                RANGE BETWEEN INTERVAL '30 days' PRECEDING
                AND CURRENT ROW
            )::NUMERIC(12, 2) AS avg_order_value_30d,

            ROW_NUMBER() OVER (
                PARTITION BY s.customer_id
                ORDER BY s.order_ts DESC, s.order_id DESC
            ) AS rn_recent

        FROM stage1_orders_total AS s
    )

    SELECT *
    FROM stage4_rolling_aov
    ORDER BY customer_id, order_ts
    LIMIT 20;
""", engine)

stage4

,customer_id,order_id,order_ts,order_total,avg_order_value_30d,rn_recent
0,00012a2ce6f8dcda20d059ce98491703,5f79b5b0931d63f1a42989eb65b9da6e,2017-11-14 16:08:26,114.74,114.74,1
1,000161a058600d5901f007fab4c27140,a44895d095d7e0702b6a162fa2dbeced,2017-07-16 09:40:32,67.41,67.41,1
2,0001fd6190edaaf884bcaf3d49edf079,316a104623542e4d75189bb372bc5f8d,2017-02-28 11:06:43,195.42,195.42,1
3,0002414f95344307404f0ace7a26f1d5,5825ce2e88d5346438686b0bba99e5ee,2017-08-16 13:09:20,179.35,179.35,1
4,000379cdec625522490c315e70c7a9fb,0ab7fb08086d4af9141453c91878ed7a,2018-04-02 13:42:17,107.01,107.01,1
5,0004164d20a9e969af783496f3408652,cd3558a10d854487b4f907e9b326a4fc,2017-04-12 08:35:12,71.80,71.80,1
6,000419c5494106c306a97b5635748086,07f6c3baf9ac86865b60f640c4f923c6,2018-03-02 17:47:40,49.40,49.40,1
7,00046a560d407e99b969756e0b10f282,8c3d752c5c02227878fae49aeaddbfd7,2017-12-18 11:08:30,166.59,166.59,1
8,00050bf6e01e69d5c0fd612f1bcfb69c,fa906f338cee30a984d0945b3832e431,2017-09-17 16:04:44,85.23,85.23,1
9,000598caf2ef4117407665ac33275130,9b961b894e797f63622137ff7eb1c1af,2018-08-11 12:14:35,1255.71,1255.71,1


In [58]:
stage5 = pd.read_sql("""
    WITH

    stage1_orders_total AS (
        -- Stage 1: orders + order_total
        SELECT
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp AS order_ts,
            o.order_purchase_timestamp::DATE AS order_date,

            COALESCE(
                SUM(
                    COALESCE(oi.price, 0)
                    + COALESCE(oi.freight_value, 0)
                ),
                0
            )::NUMERIC(12, 2) AS order_total

        FROM olist_orders AS o

        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id

        GROUP BY
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp
    ),

    stage5_cancel_rate AS (
        -- Stage 5: cancellation rate per customer
        SELECT
            customer_id,

            AVG(
                CASE
                    WHEN order_status = 'canceled'
                    THEN 1.0
                    ELSE 0.0
                END
            )::NUMERIC(6, 4) AS cancellation_rate

        FROM stage1_orders_total

        GROUP BY customer_id
    )

    SELECT *
    FROM stage5_cancel_rate
    ORDER BY cancellation_rate DESC, customer_id
    LIMIT 20;
""", engine)

stage5

,customer_id,cancellation_rate
0,002b5342c72978cf0aba6aae1f5d5293,1.0
1,00f8a4bbb515bfe8f84f9cfa45621250,1.0
2,01866d949d55c81c28d255114948b72b,1.0
3,019f5bb93ed18dd059051c3f81abe394,1.0
4,032a544bbe984490f6ff7450e3c8b0bc,1.0
5,0337719c3caba8187a276942d35347fd,1.0
6,033f5ddbb536a477e81abd786f23f452,1.0
7,0340b2de09f18a986fea9f26cc7b2d9c,1.0
8,040d94f8ba8ca26014bd6f7e8a6e0c0d,1.0
9,045d8fcab59f603256eefe86e45f0350,1.0


In [59]:
stage7 = pd.read_sql("""
    WITH

    stage6_category_revenue AS (
        -- Stage 6: revenue by customer and product category
        SELECT
            o.customer_id,

            COALESCE(
                p.product_category_name,
                '(unknown)'
            ) AS product_category_name,

            SUM(
                COALESCE(oi.price, 0)
                + COALESCE(oi.freight_value, 0)
            )::NUMERIC(12, 2) AS category_revenue

        FROM olist_orders AS o

        JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id

        LEFT JOIN olist_products AS p
            ON p.product_id = oi.product_id

        GROUP BY
            o.customer_id,
            COALESCE(
                p.product_category_name,
                '(unknown)'
            )
    ),

    stage7_preferred_category AS (
        -- Stage 7: top category per customer
        SELECT
            customer_id,
            product_category_name,
            category_revenue

        FROM (
            SELECT
                cr.*,

                ROW_NUMBER() OVER (
                    PARTITION BY cr.customer_id
                    ORDER BY
                        cr.category_revenue DESC,
                        cr.product_category_name
                ) AS rn

            FROM stage6_category_revenue AS cr
        ) AS ranked

        WHERE rn = 1
    )

    SELECT *
    FROM stage7_preferred_category
    ORDER BY category_revenue DESC
    LIMIT 20;
""", engine)

stage7

,customer_id,product_category_name,category_revenue
0,1617b1357756262bfa56ab541c47bc16,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,pcs,4681.78


In [60]:
final_features = pd.read_sql("""
    WITH

    reference_date AS (
        -- Stage 0: stable reference date
        SELECT
            (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
        FROM olist_orders
    ),

    stage1_orders_total AS (
        -- Stage 1: orders + order total
        SELECT
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp AS order_ts,
            o.order_purchase_timestamp::DATE AS order_date,

            COALESCE(
                SUM(
                    COALESCE(oi.price, 0)
                    + COALESCE(oi.freight_value, 0)
                ),
                0
            )::NUMERIC(12, 2) AS order_total

        FROM olist_orders AS o

        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id

        GROUP BY
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp
    ),

    stage2_rfm AS (
        -- Stage 2: RFM features
        SELECT
            s.customer_id,
            (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
            COUNT(*) AS frequency,
            SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
            MIN(s.order_date) AS first_order_date

        FROM stage1_orders_total AS s

        CROSS JOIN reference_date AS rd

        GROUP BY
            s.customer_id,
            rd.as_of_date
    ),

    stage3_cohort AS (
        -- Stage 3: cohort quarter
        SELECT
            r.*,

            DATE_TRUNC(
                'quarter',
                r.first_order_date
            )::DATE AS cohort_quarter

        FROM stage2_rfm AS r
    ),

    stage4_rolling_aov AS (
        -- Stage 4: rolling 30-day AOV
        SELECT
            s.customer_id,
            s.order_id,
            s.order_ts,
            s.order_total,

            AVG(s.order_total) OVER (
                PARTITION BY s.customer_id
                ORDER BY s.order_ts
                RANGE BETWEEN INTERVAL '30 days' PRECEDING
                AND CURRENT ROW
            )::NUMERIC(12, 2) AS avg_order_value_30d,

            ROW_NUMBER() OVER (
                PARTITION BY s.customer_id
                ORDER BY s.order_ts DESC, s.order_id DESC
            ) AS rn_recent

        FROM stage1_orders_total AS s
    ),

    stage5_cancel_rate AS (
        -- Stage 5: cancellation rate
        SELECT
            customer_id,

            AVG(
                CASE
                    WHEN order_status = 'canceled'
                    THEN 1.0
                    ELSE 0.0
                END
            )::NUMERIC(6, 4) AS cancellation_rate

        FROM stage1_orders_total

        GROUP BY customer_id
    ),

    stage6_category_revenue AS (
        -- Stage 6: revenue by category
        SELECT
            o.customer_id,

            COALESCE(
                p.product_category_name,
                '(unknown)'
            ) AS product_category_name,

            SUM(
                COALESCE(oi.price, 0)
                + COALESCE(oi.freight_value, 0)
            )::NUMERIC(12, 2) AS category_revenue

        FROM olist_orders AS o

        JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id

        LEFT JOIN olist_products AS p
            ON p.product_id = oi.product_id

        GROUP BY
            o.customer_id,
            COALESCE(
                p.product_category_name,
                '(unknown)'
            )
    ),

    stage7_preferred_category AS (
        -- Stage 7: preferred category
        SELECT
            customer_id,
            product_category_name,
            category_revenue

        FROM (
            SELECT
                cr.*,

                ROW_NUMBER() OVER (
                    PARTITION BY cr.customer_id
                    ORDER BY
                        cr.category_revenue DESC,
                        cr.product_category_name
                ) AS rn

            FROM stage6_category_revenue AS cr
        ) AS ranked

        WHERE rn = 1
    )

    SELECT
        c.customer_id,
        c.customer_state,

        COALESCE(
            coh.recency_days,
            9999
        ) AS recency_days,

        COALESCE(
            coh.frequency,
            0
        ) AS frequency,

        COALESCE(
            coh.monetary,
            0
        )::NUMERIC(12, 2) AS monetary,

        coh.first_order_date,
        coh.cohort_quarter,

        COALESCE(
            aov.avg_order_value_30d,
            0
        )::NUMERIC(12, 2) AS avg_order_value_30d_recent,

        COALESCE(
            cancel.cancellation_rate,
            0
        )::NUMERIC(6, 4) AS cancellation_rate,

        COALESCE(
            pc.product_category_name,
            '(none)'
        ) AS preferred_category,

        COALESCE(
            pc.category_revenue,
            0
        )::NUMERIC(12, 2) AS preferred_category_revenue

    FROM olist_customers AS c

    LEFT JOIN stage3_cohort AS coh
        ON coh.customer_id = c.customer_id

    LEFT JOIN stage4_rolling_aov AS aov
        ON aov.customer_id = c.customer_id
       AND aov.rn_recent = 1

    LEFT JOIN stage5_cancel_rate AS cancel
        ON cancel.customer_id = c.customer_id

    LEFT JOIN stage7_preferred_category AS pc
        ON pc.customer_id = c.customer_id

    ORDER BY
        monetary DESC NULLS LAST,
        c.customer_id

    LIMIT 20;
""", engine)

final_features

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0,pcs,4681.78


### Фінальний CTE pipeline

Pipeline розділений на окремі логічні етапи.

Кожен stage виконує одну конкретну трансформацію:
RFM-ознаки, cohort, rolling average order value, cancellation rate та preferred category.

Такий підхід робить SQL-запит простішим для читання, перевірки та повторного використання у feature engineering.

In [61]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS hw5_referrals CASCADE;

        CREATE TABLE hw5_referrals (
            referrer_customer_id TEXT NOT NULL
                REFERENCES olist_customers(customer_id),

            referred_customer_id TEXT NOT NULL
                REFERENCES olist_customers(customer_id),

            referred_at TIMESTAMPTZ
                NOT NULL DEFAULT NOW(),

            PRIMARY KEY (referred_customer_id),

            CHECK (
                referrer_customer_id <> referred_customer_id
            )
        );
    """))

print("hw5_referrals created!")

hw5_referrals created!


In [62]:
with engine.begin() as conn:
    conn.execute(text("""
        WITH ordered_customers AS (
            SELECT
                customer_id,

                ROW_NUMBER() OVER (
                    ORDER BY customer_id
                ) AS rn

            FROM olist_customers

            ORDER BY customer_id

            LIMIT 10
        )

        INSERT INTO hw5_referrals (
            referrer_customer_id,
            referred_customer_id
        )

        SELECT
            prev.customer_id AS referrer_customer_id,
            nxt.customer_id AS referred_customer_id

        FROM ordered_customers AS prev

        JOIN ordered_customers AS nxt
            ON nxt.rn = prev.rn + 1

        ON CONFLICT (referred_customer_id)
        DO NOTHING;
    """))

print("referral chain created!")

referral chain created!


In [63]:
referrals = pd.read_sql("""
    SELECT *
    FROM hw5_referrals
    ORDER BY
        referrer_customer_id,
        referred_customer_id;
""", engine)

referrals

,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-08 14:42:14.390180+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-08 14:42:14.390180+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-08 14:42:14.390180+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-08 14:42:14.390180+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-08 14:42:14.390180+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-08 14:42:14.390180+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-08 14:42:14.390180+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-08 14:42:14.390180+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-08 14:42:14.390180+00:00


### Referral chain

У датасеті Olist немає готової referral-таблиці, тому я створила синтетичну таблицю `hw5_referrals`.

Вона показує зв'язок між customer, який зробив referral, і customer, якого було рекомендовано.

Для прикладу використовується невеликий ланцюжок із перших 10 customers.

In [64]:
recursive_result = pd.read_sql("""
    WITH RECURSIVE referral_depth AS (

        -- ANCHOR:
        -- customers, яких ніхто не referred
        SELECT
            c.customer_id,

            1 AS depth,

            ARRAY[
                c.customer_id
            ]::TEXT[] AS path_ids,

            c.customer_id::TEXT AS path_text

        FROM olist_customers AS c

        WHERE NOT EXISTS (
            SELECT 1
            FROM hw5_referrals AS r
            WHERE r.referred_customer_id = c.customer_id
        )

        UNION ALL

        -- RECURSIVE:
        -- переходимо від referrer до referred customer
        SELECT
            r.referred_customer_id AS customer_id,

            rd.depth + 1 AS depth,

            rd.path_ids
                || r.referred_customer_id AS path_ids,

            rd.path_text
                || ' -> '
                || r.referred_customer_id AS path_text

        FROM hw5_referrals AS r

        JOIN referral_depth AS rd
            ON rd.customer_id = r.referrer_customer_id

        WHERE
            NOT r.referred_customer_id = ANY(rd.path_ids)

            AND rd.depth < 50
    )

    SELECT
        depth,
        COUNT(*) AS n_customers,
        MAX(path_text) AS sample_path

    FROM referral_depth

    GROUP BY depth

    ORDER BY depth;
""", engine)

recursive_result

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


### Recursive CTE

У цьому запиті використовується recursive CTE для проходження referral chain.

Anchor part знаходить root customers — тих, кого не було рекомендовано іншими customers.

Recursive part переходить від referrer до referred customer і збільшує `depth` на кожному рівні.

`path_text` показує повний шлях referral chain.

Для захисту від циклів використовується масив `path_ids`. Якщо customer вже є у цьому масиві, він не додається повторно.

Також використовується додатковий depth guard `depth < 50`, щоб рекурсія не могла продовжуватися безкінечно.

In [65]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP MATERIALIZED VIEW IF EXISTS
            mv_olist_customer_features CASCADE;

        CREATE MATERIALIZED VIEW mv_olist_customer_features AS

        WITH

        reference_date AS (
            SELECT
                (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
            FROM olist_orders
        ),

        stage1_orders_total AS (
            SELECT
                o.customer_id,
                o.order_id,
                o.order_status,
                o.order_purchase_timestamp AS order_ts,
                o.order_purchase_timestamp::DATE AS order_date,

                COALESCE(
                    SUM(
                        COALESCE(oi.price, 0)
                        + COALESCE(oi.freight_value, 0)
                    ),
                    0
                )::NUMERIC(12, 2) AS order_total

            FROM olist_orders AS o

            LEFT JOIN olist_order_items AS oi
                ON oi.order_id = o.order_id

            GROUP BY
                o.customer_id,
                o.order_id,
                o.order_status,
                o.order_purchase_timestamp
        ),

        stage2_rfm AS (
            SELECT
                s.customer_id,

                (
                    rd.as_of_date - MAX(s.order_date)
                )::INTEGER AS recency_days,

                COUNT(*) AS frequency,

                SUM(
                    s.order_total
                )::NUMERIC(12, 2) AS monetary,

                MIN(s.order_date) AS first_order_date

            FROM stage1_orders_total AS s

            CROSS JOIN reference_date AS rd

            GROUP BY
                s.customer_id,
                rd.as_of_date
        ),

        stage3_cohort AS (
            SELECT
                r.*,

                DATE_TRUNC(
                    'quarter',
                    r.first_order_date
                )::DATE AS cohort_quarter

            FROM stage2_rfm AS r
        ),

        stage4_rolling_aov AS (
            SELECT
                s.customer_id,
                s.order_id,
                s.order_ts,
                s.order_total,

                AVG(s.order_total) OVER (
                    PARTITION BY s.customer_id
                    ORDER BY s.order_ts

                    RANGE BETWEEN
                        INTERVAL '30 days' PRECEDING
                        AND CURRENT ROW
                )::NUMERIC(12, 2) AS avg_order_value_30d,

                ROW_NUMBER() OVER (
                    PARTITION BY s.customer_id
                    ORDER BY
                        s.order_ts DESC,
                        s.order_id DESC
                ) AS rn_recent

            FROM stage1_orders_total AS s
        ),

        stage5_cancel_rate AS (
            SELECT
                customer_id,

                AVG(
                    CASE
                        WHEN order_status = 'canceled'
                        THEN 1.0
                        ELSE 0.0
                    END
                )::NUMERIC(6, 4) AS cancellation_rate

            FROM stage1_orders_total

            GROUP BY customer_id
        ),

        stage6_category_revenue AS (
            SELECT
                o.customer_id,

                COALESCE(
                    p.product_category_name,
                    '(unknown)'
                ) AS product_category_name,

                SUM(
                    COALESCE(oi.price, 0)
                    + COALESCE(oi.freight_value, 0)
                )::NUMERIC(12, 2) AS category_revenue

            FROM olist_orders AS o

            JOIN olist_order_items AS oi
                ON oi.order_id = o.order_id

            LEFT JOIN olist_products AS p
                ON p.product_id = oi.product_id

            GROUP BY
                o.customer_id,
                COALESCE(
                    p.product_category_name,
                    '(unknown)'
                )
        ),

        stage7_preferred_category AS (
            SELECT
                customer_id,
                product_category_name,
                category_revenue

            FROM (
                SELECT
                    cr.*,

                    ROW_NUMBER() OVER (
                        PARTITION BY cr.customer_id
                        ORDER BY
                            cr.category_revenue DESC,
                            cr.product_category_name
                    ) AS rn

                FROM stage6_category_revenue AS cr
            ) AS ranked

            WHERE rn = 1
        )

        SELECT
            c.customer_id,
            c.customer_state,

            COALESCE(
                coh.recency_days,
                9999
            ) AS recency_days,

            COALESCE(
                coh.frequency,
                0
            ) AS frequency,

            COALESCE(
                coh.monetary,
                0
            )::NUMERIC(12, 2) AS monetary,

            coh.first_order_date,
            coh.cohort_quarter,

            COALESCE(
                aov.avg_order_value_30d,
                0
            )::NUMERIC(12, 2) AS avg_order_value_30d_recent,

            COALESCE(
                cancel.cancellation_rate,
                0
            )::NUMERIC(6, 4) AS cancellation_rate,

            COALESCE(
                pc.product_category_name,
                '(none)'
            ) AS preferred_category,

            COALESCE(
                pc.category_revenue,
                0
            )::NUMERIC(12, 2) AS preferred_category_revenue

        FROM olist_customers AS c

        LEFT JOIN stage3_cohort AS coh
            ON coh.customer_id = c.customer_id

        LEFT JOIN stage4_rolling_aov AS aov
            ON aov.customer_id = c.customer_id
           AND aov.rn_recent = 1

        LEFT JOIN stage5_cancel_rate AS cancel
            ON cancel.customer_id = c.customer_id

        LEFT JOIN stage7_preferred_category AS pc
            ON pc.customer_id = c.customer_id;
    """))

print("Materialized View created!")

Materialized View created!


In [66]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE UNIQUE INDEX
            idx_mv_olist_customer_features_customer_id

        ON mv_olist_customer_features (
            customer_id
        );
    """))

print("Unique index created!")

Unique index created!


In [67]:
mv_check = pd.read_sql("""
    SELECT *
    FROM mv_olist_customer_features
    ORDER BY monetary DESC NULLS LAST, customer_id
    LIMIT 5;
""", engine)

mv_check

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66


In [68]:
with engine.connect().execution_options(
    isolation_level="AUTOCOMMIT"
) as conn:

    conn.execute(text("""
        REFRESH MATERIALIZED VIEW CONCURRENTLY
        mv_olist_customer_features;
    """))

print("Concurrent refresh completed!")

Concurrent refresh completed!


### Materialized View

Я створила Materialized View `mv_olist_customer_features`, яка зберігає готові customer-level features з CTE pipeline.

Materialized View корисна тим, що складні features не потрібно перераховувати при кожному запиті.

Для `customer_id` створено `UNIQUE INDEX`. Він потрібен для використання `REFRESH MATERIALIZED VIEW CONCURRENTLY`.

Concurrent refresh дозволяє оновити дані Materialized View без повного блокування читання з неї.

In [69]:
explain_cte = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    WITH

    reference_date AS (
        SELECT
            (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
        FROM olist_orders
    ),

    stage1_orders_total AS (
        SELECT
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp AS order_ts,
            o.order_purchase_timestamp::DATE AS order_date,

            COALESCE(
                SUM(
                    COALESCE(oi.price, 0)
                    + COALESCE(oi.freight_value, 0)
                ),
                0
            )::NUMERIC(12, 2) AS order_total

        FROM olist_orders AS o

        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id

        GROUP BY
            o.customer_id,
            o.order_id,
            o.order_status,
            o.order_purchase_timestamp
    ),

    stage2_rfm AS (
        SELECT
            s.customer_id,

            (
                rd.as_of_date - MAX(s.order_date)
            )::INTEGER AS recency_days,

            COUNT(*) AS frequency,

            SUM(
                s.order_total
            )::NUMERIC(12, 2) AS monetary,

            MIN(s.order_date) AS first_order_date

        FROM stage1_orders_total AS s

        CROSS JOIN reference_date AS rd

        GROUP BY
            s.customer_id,
            rd.as_of_date
    )

    SELECT *
    FROM stage2_rfm

    ORDER BY monetary DESC NULLS LAST

    LIMIT 10;
""", engine)

explain_cte

,QUERY PLAN
0,Limit (cost=33552.08..33552.10 rows=10 width=...
1,"Buffers: shared hit=5966, temp read=3354 wri..."
2,-> Sort (cost=33552.08..33552.58 rows=200 ...
3,Sort Key: stage2_rfm.monetary DESC NUL...
4,Sort Method: top-N heapsort Memory: 27kB
5,"Buffers: shared hit=5966, temp read=33..."
6,-> Subquery Scan on stage2_rfm (cost...
7,"Buffers: shared hit=5966, temp r..."
8,-> HashAggregate (cost=33542.2...
9,"Group Key: o.customer_id, ..."


### EXPLAIN ANALYZE для CTE pipeline

Я використала `EXPLAIN (ANALYZE, BUFFERS)`, щоб подивитися, як PostgreSQL виконує CTE pipeline.

У плані можна побачити операції JOIN, GROUP BY, сортування та агрегації.

Також EXPLAIN показує фактичний час виконання запиту і використання buffers.

In [70]:
explain_mv = pd.read_sql("""
    EXPLAIN (ANALYZE, BUFFERS)

    WITH target_customer AS (
        SELECT
            customer_id

        FROM olist_customers

        ORDER BY customer_id

        LIMIT 1
    )

    SELECT
        mv.*

    FROM mv_olist_customer_features AS mv

    JOIN target_customer AS t
        ON t.customer_id = mv.customer_id;
""", engine)

explain_mv

,QUERY PLAN
0,Nested Loop (cost=0.83..8.98 rows=1 width=158...
1,Buffers: shared hit=5 read=3
2,-> Limit (cost=0.42..0.55 rows=1 width=32)...
3,Buffers: shared hit=4
4,-> Index Only Scan using olist_custom...
5,Heap Fetches: 1
6,Buffers: shared hit=4
7,-> Index Scan using idx_mv_olist_customer_f...
8,Index Cond: (customer_id = olist_custo...
9,Buffers: shared hit=1 read=3


### EXPLAIN ANALYZE для Materialized View

У цьому запиті я перевіряю lookup одного customer через Materialized View.

Для `customer_id` раніше був створений `UNIQUE INDEX`, тому PostgreSQL може використовувати індекс для швидкого пошуку потрібного рядка.

Я порівнюю цей план із повним CTE pipeline, щоб оцінити різницю у складності та часі виконання.

In [71]:
print("=== CTE PIPELINE ===")
print("\n".join(explain_cte.iloc[:, 0].astype(str)))

print("\n=== MATERIALIZED VIEW ===")
print("\n".join(explain_mv.iloc[:, 0].astype(str)))

=== CTE PIPELINE ===
Limit  (cost=33552.08..33552.10 rows=10 width=64) (actual time=537.732..539.304 rows=10 loops=1)
  Buffers: shared hit=5966, temp read=3354 written=4093
  ->  Sort  (cost=33552.08..33552.58 rows=200 width=64) (actual time=537.730..539.301 rows=10 loops=1)
        Sort Key: stage2_rfm.monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=5966, temp read=3354 written=4093
        ->  Subquery Scan on stage2_rfm  (cost=33542.25..33547.75 rows=200 width=64) (actual time=424.872..520.916 rows=99441 loops=1)
              Buffers: shared hit=5966, temp read=3354 written=4093
              ->  HashAggregate  (cost=33542.25..33545.75 rows=200 width=68) (actual time=424.870..512.230 rows=99441 loops=1)
                    Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))
                    Batches: 21  Memory Usage: 8265kB  Disk Usage: 7400kB
                    Buffers: shared hit=5966, te

## Reflection

Найбільш дорогим у моєму CTE pipeline виявилося агрегування даних для розрахунку customer-level features. У `EXPLAIN ANALYZE` видно `GroupAggregate`, `HashAggregate`, `Sort`, `Parallel Hash Right Join` та `Seq Scan`. Повний CTE-запит виконався приблизно за **1413.639 ms**. Особливо помітно, що `HashAggregate` використовував не тільки пам'ять, але й диск (`Disk Usage: 7400kB`). Також один із `Sort` виконувався як `external sort` і використовував приблизно `8040kB` диску. Це означає, що JOIN, GROUP BY та сортування великої кількості рядків потребують значно більше ресурсів.

У цьому випадку Materialized View є корисною, тому що вже готові features не потрібно перераховувати при кожному lookup. Запит до Materialized View виконався лише за **0.151 ms**, що набагато швидше за повний CTE pipeline. У плані видно `Index Only Scan` для таблиці customers та `Index Scan` по індексу `idx_mv_olist_customer_features_customer_id`. Для такого історичного датасету я б обрала refresh один раз на добу або on-demand.

Звичайний `VIEW` був би кращим, якщо дані повинні завжди бути максимально актуальними, а сам запит не є дорогим для повторного виконання.

Recursive CTE завершується завдяки двом захистам. `path_ids` перевіряє, чи customer уже був у шляху, а `depth < 50` обмежує максимальну глибину. При циклі `A → B → A` повторний `A` не буде доданий, тому рекурсія зупиниться. :chatgpt-content-reference{index="1"}

Для цього pipeline найважливішим trade-off я вважаю баланс між **freshness та latency**. Materialized View може містити трохи старіші дані, але значно прискорює читання і зменшує вартість повторних обчислень.